# Snake

In [ ]:
import environments_fully_observable 
import environments_partially_observable
import numpy as np
from  tqdm import trange
import matplotlib.pyplot as plt
import random
import tensorflow as tf
import os

from agent import Agent
import algorithms.actor_critic as ac

tf.random.set_seed(0)
random.seed(0)
np.random.seed(0)

## Environment definition

In [ ]:
%matplotlib inline
# function to standardize getting an env for the whole notebook

N = 1000

def get_env(n=N):
    # n is the number of boards that you want to simulate parallely
    # size is the size of each board, also considering the borders
    # mask for the partially observable, is the size of the local neighborhood
    size = 7
    e = environments_fully_observable.OriginalSnakeEnvironment(n, size)
    # or environments_partially_observable.OriginalSnakeEnvironment(n, size, 2)
    return e
env_ = get_env()

GAMMA = .9
ITERATIONS = 20000

SAVE_FREQUENCY = 500
MODELS_PATH = "models"


In [ ]:
fig,axs=plt.subplots(1,min(len(env_.boards), 5), figsize=(10,3))
for ax, board in zip(axs, env_.boards):
    ax.get_yaxis().set_visible(False)
    ax.get_xaxis().set_visible(False)
    ax.imshow(board, origin="lower")

## Model

In [ ]:
optimizer = tf.keras.optimizers.Adam(1e-4)

# define the models that you need ()
logic = ac.create_logic(state_shape=env_.to_state().shape[1:], action_dim=4, optimizer=optimizer, gamma=GAMMA)

agent = Agent(algorithm_logic=logic, algorithm_name="actor_critic_separed_loss", algorithm_path=MODELS_PATH, save_frequency=500) #wrapper for the agent that interacts with the environment, it will call the logic to get the action and to train the model

## Training

In [ ]:
rewards_history = []

progress_bar = trange(ITERATIONS)

for iteration in progress_bar:
    # get current state of the boards
    state = env_.to_state()
    """ 
    tensor of actions, consider that
        UP = 0
        RIGHT = 1
        DOWN = 2
        LEFT = 3
    """

    #agent extract action probabilities from the state
    #sample action
    actions, _ = agent.get_action(state, training=True)

    rewards = env_.move(actions)
    new_state = tf.cast(env_.to_state(), dtype=tf.float32)

    done = tf.cast(rewards < 0, tf.float32) # if reward is negative then it could only be due to the snale hitting the wall or eating itself, so the episode ends

    # calculate the loss of whichever algorithm you have picked
    loss = agent.train_step(state, actions, rewards, new_state, done)

    avg_reward = tf.reduce_mean(rewards).numpy()
    rewards_history.append(avg_reward)

    if iteration % 10 == 0:
        recent_avg = np.mean(rewards_history[-100:]) if rewards_history else 0
        progress_bar.set_description(f"Loss: {loss:.4f} | Avg Reward: {recent_avg:.3f}")

    if (iteration + 1) % SAVE_FREQUENCY == 0:
        agent.save_results_plots()

 ### Random policy reward
 
Just a baseline (not the one you are supposed to develop)

In [ ]:
random_env = get_env(100)
random_rewards = []

for _ in trange(1000):
    probs = tf.convert_to_tensor([[.25]*4]*random_env.n_boards)
    #sample actions
    actions =  tf.random.categorical(tf.math.log(probs), 1, dtype=tf.int32)
    # MDP update
    rewards = random_env.move(actions)
    random_rewards.append(np.mean(rewards))